## Dia 6 — Limpar preservando o que aconteceu

### Situação

Uma loja exportou seus pedidos, mas o arquivo contém problemas que impedem uma soma confiável dos itens vendidos. Você recebeu a tarefa de separar os registros utilizáveis e documentar o destino de cada linha, sem inventar valores para completar o cadastro.

### Dados disponíveis

Use `pedidos_sujos.csv`, com as colunas `pedido`, `quantidade` e `preco`. O campo `pedido` identifica o pedido. Neste cadastro, a quantidade deve ser um inteiro maior ou igual a zero, e o preço deve ser um número maior ou igual a zero. Os preços válidos usam vírgula decimal. Há uma duplicata exata, valores ausentes e valores inválidos. Quantidade zero é permitida.

### Sua tarefa

1. Leia inicialmente todas as colunas como texto, preservando os valores originais para conferência.
2. Converta quantidade e preço de forma explícita. Diferencie valor ausente, formato inválido e valor que viola as regras do cadastro.
3. Remova apenas a repetição exata de um pedido. Separe os outros registros problemáticos para revisão, sem substituir ausências por zero nem corrigir valores por suposição.
4. Produza uma tabela de registros válidos e uma auditoria que identifique a linha original, o problema e o destino dado a ela. Demonstre que todas as linhas originais foram contabilizadas, incluindo a duplicata removida.
5. Some as quantidades somente dos registros válidos. Aplique novamente o tratamento à saída válida e confira se seu conteúdo permanece igual.

### Entrega

Apresente o código, a tabela limpa, a auditoria, o balanço das linhas e o total de itens dos registros válidos.

### Confira seu avanço

- [ ] Consigo identificar o destino de cada linha original, inclusive da duplicata.
- [ ] Ausências continuam distintas de zero, e preços inválidos aparecem na auditoria.
- [ ] Reprocessar a saída válida não modifica seus registros.

### Interpretação

1. O que seria afirmado indevidamente ao preencher uma quantidade ausente com zero?
2. Por que o total calculado nos registros válidos pode não representar todas as vendas da loja?

### Leituras de apoio

Análise de Dados: Da Teoria à Prática, pp. 10-11; roadmap E1.


## Minha estratégia

Explique como pretende abordar o problema.

## Resolução

Os arquivos de dados estão nesta pasta. Se necessário, confira o diretório de execução com `Path.cwd()`. Acrescente células conforme precisares.

Use `pedidos_sujos.csv`, com as colunas `pedido`, `quantidade` e `preco`. O campo `pedido` identifica o pedido. Neste cadastro, a quantidade deve ser um inteiro maior ou igual a zero, e o preço deve ser um número maior ou igual a zero. Os preços válidos usam vírgula decimal. Há uma duplicata exata, valores ausentes e valores inválidos. Quantidade zero é permitida.

In [31]:
# 1. Leia inicialmente todas as colunas como texto, preservando os valores originais para conferência.

import pandas as pd 

dados_pedidos = pd.read_csv("pedidos_sujos.csv", dtype=str)
dados_pedidos

,pedido,quantidade,preco
0,A1,2,"10,50"
1,A2,NaN,"8,00"
2,A3,-1,"15,00"
3,A1,2,"10,50"
4,A4,3,erro
5,A5,0,"5,00"


In [32]:
# 2. Converta quantidade e preço de forma explícita. Diferencie valor ausente, 
# formato inválido e valor que viola as regras do cadastro.

# ====== Quantidade: identificar a origem de cada problema ======
quantidade_original = dados_pedidos['quantidade'].astype('string').str.strip()
quantidade_numerica = pd.to_numeric(quantidade_original, errors='coerce')

# ------ O campo já estava vazio no dado original ------
quantidade_ausente = (
    quantidade_original.isna() | quantidade_original.eq('')
)

# ------ O campo tinha conteúdo, mas a conversão falhou ------
quantidade_invalida = (
    ~quantidade_ausente & quantidade_numerica.isna()
)

# ------ Virou número, mas é negativo ou tem parte fracionária ------
quantidade_violacao = quantidade_numerica.notna() & (
    quantidade_numerica.lt(0) | quantidade_numerica.mod(1).ne(0)
)

# ====== Conferência da quantidade ======

conferencia_quantidade = dados_pedidos[['pedido', 'quantidade']].assign(
    quantidade_convertida=quantidade_numerica,
    ausente=quantidade_ausente,
    formato_invalido=quantidade_invalida,
    violacao_regra=quantidade_violacao,
)

display(conferencia_quantidade)

# ====== Preço: identificar a origem de cada problema ======
preco_original = dados_pedidos["preco"].astype('string').str.strip()
preco_numerico = pd.to_numeric(preco_original.str.replace(",", "."), errors="coerce")

# ------ O campo já estava vazio no dado original ------
preco_ausente = (
    preco_original.isna() | preco_original.eq('')
)

# ------ O campo tinha conteúdo, mas a conversão falhou ------
preco_invalida = (
    ~preco_ausente & preco_numerico.isna()
)

# ------ Virou número, mas é negativo ------
preco_violacao = preco_numerico.notna() & (
    preco_numerico.lt(0)
)

conferencia_preco = dados_pedidos[['pedido', 'preco']].assign(
    preco_convertida=preco_numerico,
    ausente=preco_ausente,
    formato_invalido=preco_invalida,
    violacao_regra=preco_violacao,
)

display(conferencia_preco)

,pedido,quantidade,quantidade_convertida,ausente,formato_invalido,violacao_regra
0,A1,2,2,False,False,False
1,A2,NaN,<NA>,True,False,False
2,A3,-1,-1,False,False,True
3,A1,2,2,False,False,False
4,A4,3,3,False,False,False
5,A5,0,0,False,False,False


,pedido,preco,preco_convertida,ausente,formato_invalido,violacao_regra
0,A1,"10,50",10.5,False,False,False
1,A2,"8,00",8.0,False,False,False
2,A3,"15,00",15.0,False,False,False
3,A1,"10,50",10.5,False,False,False
4,A4,erro,<NA>,False,True,False
5,A5,"5,00",5.0,False,False,False


In [33]:
# 3. Remova apenas a repetição exata de um pedido. Separe os outros registros problemáticos para revisão, 
# sem substituir ausências por zero nem corrigir valores por suposição.


# ====== Remover a repetição exata ======

duplicata_exata = dados_pedidos.duplicated(keep='first')

duplicata = dados_pedidos.loc[duplicata_exata]
dados_pedidos_tratado = dados_pedidos.loc[~duplicata_exata]

# ====== Separar os registros para revisão ======

tem_problema = (
    quantidade_ausente | quantidade_invalida | quantidade_violacao
    | preco_ausente | preco_invalida | preco_violacao
)

pedidos_para_revisao = dados_pedidos.loc[
    ~duplicata_exata & tem_problema
]

display(duplicata)
display(pedidos_para_revisao)


,pedido,quantidade,preco
3,A1,2,"10,50"


,pedido,quantidade,preco
1,A2,NaN,"8,00"
2,A3,-1,"15,00"
4,A4,3,erro


In [34]:
# 4. Produza uma tabela de registros válidos e uma auditoria que identifique a linha original,
# o problema e o destino dado a ela. Demonstre que todas as linhas originais foram contabilizadas,
# incluindo a duplicata removida.

# ====== Montar a tabela de pedidos válidos ======

indices_excluidos = pedidos_para_revisao.index.union(duplicata.index)

tab_validos = dados_pedidos.loc[
    ~dados_pedidos.index.isin(indices_excluidos)
].copy()

# ------ Usar os valores numéricos já conferidos no ponto 2 ------

tab_validos['quantidade'] = quantidade_numerica.loc[tab_validos.index].astype(
    'Int64'
)
tab_validos['preco'] = preco_numerico.loc[tab_validos.index].astype('Float64')

display(tab_validos)

# ====== Auditoria das linhas originais ======

auditoria = dados_pedidos.copy()
auditoria['problema'] = ''
auditoria['destino'] = 'válido'

auditoria.loc[pedidos_para_revisao.index, 'destino'] = 'revisão'
auditoria.loc[duplicata.index, 'destino'] = 'duplicata removida'

# ------ Registrar todos os problemas da linha enviada para revisão ------

motivos = {
    'quantidade ausente': quantidade_ausente,
    'quantidade com formato inválido': quantidade_invalida,
    'quantidade viola a regra': quantidade_violacao,
    'preço ausente': preco_ausente,
    'preço com formato inválido': preco_invalida,
    'preço viola a regra': preco_violacao,
}

for motivo, mascara in motivos.items():
    linhas = mascara & auditoria['destino'].eq('revisão')
    auditoria.loc[linhas, 'problema'] += motivo + ' | '

auditoria['problema'] = auditoria['problema'].str.rstrip(' |')
auditoria.loc[duplicata.index, 'problema'] = 'repetição exata'

auditoria = auditoria.rename_axis('indice_original').reset_index()
display(auditoria)





,pedido,quantidade,preco
0,A1,2,10.5
5,A5,0,5.0


,indice_original,pedido,quantidade,preco,problema,destino
0,0,A1,2,"10,50",,válido
1,1,A2,NaN,"8,00",quantidade ausente,revisão
2,2,A3,-1,"15,00",quantidade viola a regra,revisão
3,3,A1,2,"10,50",repetição exata,duplicata removida
4,4,A4,3,erro,preço com formato inválido,revisão
5,5,A5,0,"5,00",,válido


In [35]:
# 5. Some as quantidades somente dos registros válidos. Aplique novamente o tratamento à saída válida e 
# confira se seu conteúdo permanece igual.

# ====== Somar as quantidades dos pedidos válidos ======

total_itens = tab_validos['quantidade'].sum()
print(f'Total de itens nos pedidos válidos: {total_itens}')

# ====== Reaplicar o tratamento à tabela válida ======

reprocessados = tab_validos.drop_duplicates().copy()

# ------ Converter quantidade e preço novamente ------

quantidade_reprocessada = pd.to_numeric(
    reprocessados['quantidade'].astype('string').str.strip(),
    errors='coerce',
)

preco_normalizado = (
    reprocessados['preco']
    .astype('string')
    .str.strip()
    .str.replace(',', '.', regex=False)
)
preco_reprocessado = pd.to_numeric(preco_normalizado, errors='coerce')

# ------ Manter quantidades inteiras e preços não negativos ------

continuam_validos = (
    quantidade_reprocessada.notna()
    & quantidade_reprocessada.ge(0)
    & quantidade_reprocessada.mod(1).eq(0)
    & preco_reprocessado.notna()
    & preco_reprocessado.ge(0)
)

reprocessados = reprocessados.loc[continuam_validos].copy()

reprocessados['quantidade'] = quantidade_reprocessada.loc[
    reprocessados.index
].astype('Int64')
reprocessados['preco'] = preco_reprocessado.loc[
    reprocessados.index
].astype('Float64')

# ====== Conferir se a segunda aplicação mudou a tabela ======

mesmo_resultado = tab_validos.equals(reprocessados)
print(f'A tabela permaneceu igual após o reprocessamento? {mesmo_resultado}')

Total de itens nos pedidos válidos: 2
A tabela permaneceu igual após o reprocessamento? True


## Resultados e interpretação

1. O que seria afirmado indevidamente ao preencher uma quantidade ausente com zero?

R: haveria uma mudança de significado. Pois a quantidade zero é válida, e não podemos afirmar que o pedido não tinha itens

2. Por que o total calculado nos registros válidos pode não representar todas as vendas da loja?

R: Pois os registros inválidos podem ser pedidos reais. Algum problema na armazenagem da informação não o exclui.

## Conferência dos critérios

Registre as evidências de conclusão e as dúvidas que restaram.